# Task 2 - Natural Gas Storage Contract Valuation

## JP Morgan Chase & Co. Quantitative Research Virtual Experience

**Author:** Ruth Wanjiru

### Objective
Develop a pricing model that estimates the value of a natural gas storage contract using forecasted natural gas prices, storage costs, operational costs, and storage capacity constraints. 

## Business Background

Natural gas storage allows market participants to purchase gas when prices are low and sell it when prices are higher. The profitability of a storage contract depends on market prices, storage duration, operating costs, and storage capacity. This notebook develops a pricing model that estimates the financial value of a storage contract under different market scenarios using the forecasted prices generated in Task 1.


## Project Objectives

* Build reusable pricing functions for natural gas storage contracts.
* Estimate prices for any historical or forecasted date.
* Calculate the value of a storage contract using user-defined inputs.
* Evaluate multiple pricing scenarios.
* Determine whether a storage contract is profitable under different market conditions.


In [1]:
# Import libraries
import pandas as pd
import numpy as np

In [2]:
# Load the dataset
df = pd.read_csv("Nat_Gas.csv")

In [3]:
df.head()

,Dates,Prices
0,10/31/20,10.1
1,11/30/20,10.3
2,12/31/20,11.0
3,1/31/21,10.9
4,2/28/21,10.9


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Dates   48 non-null     str    
 1   Prices  48 non-null     float64
dtypes: float64(1), str(1)
memory usage: 900.0 bytes


In [5]:
print(df.columns)

Index(['Dates', 'Prices'], dtype='str')


In [6]:
# Convert Dates column
df["Dates"] = pd.to_datetime(df["Dates"], format="%m/%d/%y")

In [7]:
# Set the index
df.set_index("Dates", inplace=True)

In [8]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 48 entries, 2020-10-31 to 2024-09-30
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Prices  48 non-null     float64
dtypes: float64(1)
memory usage: 768.0 bytes


In [9]:
def load_data(file_path="Nat_Gas.csv"):
    """
    Load and prepare the natural gas price dataset.
    """
    df = pd.read_csv(file_path)
    df["Dates"] = pd.to_datetime(df["Dates"], format="%m/%d/%y")
    df.set_index("Dates", inplace=True)
    return df

In [10]:
# Create daily historical prices from monthly historical prices 
def create_daily_history(df):
    """
    Convert monthly prices to daily prices using linear interpolation.
    """
    daily_index = pd.date_range(
        start=df.index.min(),
        end=df.index.max(),
        freq="D"
    )

    df_daily = df.reindex(daily_index)
    df_daily["Prices"] = df_daily["Prices"].interpolate(method="linear")

    return df_daily

In [11]:
# Create monthly forecast prices 
def forecast_prices(df, months=12):
    """
    Forecast monthly prices using the average monthly trend
    and historical seasonal effects.
    """

    monthly_change = df["Prices"].diff().mean()

    monthly_average = df.groupby(df.index.month)["Prices"].mean()
    overall_average = df["Prices"].mean()
    seasonal_effect = monthly_average - overall_average

    forecast_dates = pd.date_range(
        start=df.index.max() + pd.offsets.MonthEnd(),
        periods=months,
        freq="ME"
    )

    forecast_values = []

    last_price = df["Prices"].iloc[-1]

    for date in forecast_dates:
        seasonality = seasonal_effect.loc[date.month]
        next_price = last_price + monthly_change + seasonality

        forecast_values.append(round(next_price, 2))
        last_price = next_price

    forecast_df = pd.DataFrame(
        {"Prices": forecast_values},
        index=forecast_dates
    )

    return forecast_df

In [12]:
# Create full price table with all prices
def build_full_price_table():
    """
    Combine historical daily prices with forecasted daily prices.
    """

    df = load_data()

    historical_daily = create_daily_history(df)

    forecast_df = forecast_prices(df)

    forecast_daily_index = pd.date_range(
        start=forecast_df.index.min(),
        end=forecast_df.index.max(),
        freq="D"
    )

    forecast_daily = forecast_df.reindex(forecast_daily_index)

    forecast_daily["Prices"] = forecast_daily["Prices"].interpolate(
        method="linear"
    )

    full_prices = pd.concat([historical_daily, forecast_daily])

    return full_prices


In [13]:
FULL_PRICES = build_full_price_table()

In [14]:
# Create function showing price for any date 
def estimate_price(date):
    """
    Estimate the natural gas price for any supported date.
    """

    date = pd.to_datetime(date)

    if date < FULL_PRICES.index.min() or date > FULL_PRICES.index.max():
        raise ValueError(
            f"Date must be between "
            f"{FULL_PRICES.index.min().date()} and "
            f"{FULL_PRICES.index.max().date()}."
        )

    return round(FULL_PRICES.loc[date, "Prices"], 2)


In [15]:
# Create price contract function
def price_contract(
    injection_date,
    withdrawal_date,
    volume,
    storage_cost_per_month,
    injection_cost,
    withdrawal_cost,
    max_storage,
):
    """
    Calculate the value of a natural gas storage contract.
    """

    if volume > max_storage:
        raise ValueError("Volume exceeds maximum storage capacity.")

    buy_price = estimate_price(injection_date)
    sell_price = estimate_price(withdrawal_date)

    purchase_cost = buy_price * volume
    sales_revenue = sell_price * volume

    injection_date = pd.to_datetime(injection_date)
    withdrawal_date = pd.to_datetime(withdrawal_date)

    storage_months = (
        (withdrawal_date.year - injection_date.year) * 12
        + withdrawal_date.month
        - injection_date.month
    )

    total_storage_cost = storage_months * storage_cost_per_month
    total_operating_cost = injection_cost + withdrawal_cost

    contract_value = (
        sales_revenue
        - purchase_cost
        - total_storage_cost
        - total_operating_cost
    )

    return round(contract_value, 2)

In [16]:
# Testing the price contract function - Scenario 1
price_contract(
    injection_date="2024-06-30",
    withdrawal_date= "2024-12-31",
    volume=1_000_000,
    storage_cost_per_month=100000,
    injection_cost=10000,
    withdrawal_cost=10000,
    max_storage=2000000
)

np.float64(-60000.0)

In [17]:
# Testing the price contract function - Scenario 2
price_contract(
    injection_date="2024-03-31",
    withdrawal_date= "2024-11-30",
    volume=500_000,
    storage_cost_per_month=75000,
    injection_cost=10000,
    withdrawal_cost=10000,
    max_storage=2000000
)

np.float64(-1205000.0)

In [18]:
# Testing the price contract function - Scenario 3
price_contract(
    injection_date="2024-01-31",
    withdrawal_date= "2024-09-30",
    volume=1_500_000,
    storage_cost_per_month=50000,
    injection_cost=15000,
    withdrawal_cost=15000,
    max_storage=2000000
)

np.float64(-1630000.0)

##### Results Interpretation
Three sample contract scenarios were tested using the pricing model. All three scenarios produced negative contract 
values (-60,000, -1,205,000, and -1,630,000), indicating that the total costs of storing and handling the natural gas exceeded
the potential profit from buying and selling the commodity over the selected periods. The first scenario resulted in the 
smallest loss, suggesting that it had the most favorable balance between the price spread and the associated costs. The second
and third scenarios produced larger losses, primarily due to higher storage costs, longer storage periods, larger transaction 
volumes, or a combination of these factors. These results demonstrate that the pricing model correctly incorporates the key 
cash flows of a natural gas storage contract, including purchase costs, sales revenue, storage costs, and operational costs. 
The model is therefore able to distinguish between economically attractive and unattractive storage opportunities based on the
input assumptions. 

In [19]:
# Testing price contract function for profitability
price_contract(
    injection_date="2024-06-30",
    withdrawal_date="2024-12-31",
    volume=1_000_000,
    storage_cost_per_month=10000,
    injection_cost=1000,
    withdrawal_cost=1000,
    max_storage=2000000
)

np.float64(498000.0)

##### Profitability Validation
An additional validation test was performed using lower storage and operational costs while keeping the contract volume 
and trading dates unchanged. Under these assumptions, the pricing model returned a positive contract value of 498,000, 
indicating that the expected revenue from selling the stored natural gas exceeded the total purchase, storage, and operational
costs. This result confirms that the pricing model behaves as expected by distinguishing between profitable and unprofitable 
storage opportunities based on the economic assumptions provided. It demonstrates that the model can support trading decisions
by identifying contracts with positive expected value while rejecting those that are not economically . """

# Conclusion

This notebook developed a reusable pricing model for natural gas storage contracts using forecasted market prices. The model incorporates purchase and selling prices, storage duration, storage costs, operational costs, and storage capacity constraints to estimate contract value under different scenarios.

Testing demonstrated that contract profitability depends on both market price movements and operating costs. While some scenarios resulted in losses, the profitability analysis showed that favorable market conditions can generate positive returns. The reusable pricing functions created in this notebook form the foundation for practical storage contract valuation and risk assessment.
